Can Large Language Models Mine Interpretable Financial Factors More Effectively? A Neural-Symbolic Factor Mining Agent Model

link : https://aclanthology.org/2024.findings-acl.233.pdf

Authors  :
Zhiwei Li1, Ran Song2, Caihong Sun1 ∗, Wei Xu1,3, Zhengtao Yu2, Ji-Rong Wen1,41 School of Information, Renmin University of China, Beijing, China2 Faculty of Information Engineering and Automation,
Kunming University of Science and Technology, Kunming, China3 School of Smart Governance, Renmin University of China, Beijing, China4 Gaoling School of Artificial Intelligence, Renmin University of China, Beijing, China{lizhiwei2,chsun,weixu,jrwen}@ruc.edu.cn,song_ransr@163.com, ztyu@hotmail.com

## 한 줄 요약

LLM에게 기존 알파 수식을 예시로 주고 새 수식을 만들게 하되, **예시를 서로 다른 클러스터에서 골라 다양성을 확보하고(CSS)**, **성능이 좋아진 순서의 수식 경로를 함께 보여줘 개선 방향을 유도하는(CoE)** 팩터 마이닝 프레임워크 FAMA를 제안한 논문입니다. ACL 2024 Findings에 실렸습니다.

---

## 1. 문제의식

기존 팩터 마이닝은 두 갈래입니다.

**기호(symbolic) 방식**은 GP 같은 기호 회귀로 수식을 탐색합니다. 결과가 수식이라 읽을 수 있지만, 가능한 수식 조합이 너무 많아 탐색이 비효율적입니다.

**신경망(neural) 방식**은 FactorVAE, ALSTM처럼 원시 데이터에서 신경망이 직접 특징을 뽑습니다. 예측력은 좋지만 무엇을 근거로 예측하는지 알 수 없습니다.

저자들은 LLM이 이 둘을 잇는 다리가 될 수 있다고 봅니다. LLM은 신경망이지만 출력은 수식 텍스트이기 때문입니다. 그런데 공개된 알파 수식을 그냥 예시로 넣고 새 수식을 만들라고 하면 두 문제가 생깁니다. 첫째, 예시들이 서로 비슷해서 LLM이 비슷한 형태의 수식만 반복 생성합니다(동질성). 둘째, 성능이 나쁜 예시가 섞여 있으면 LLM이 좋은 방향을 찾지 못합니다. CSS가 첫째를, CoE가 둘째를 해결하려는 장치입니다.

---

## 2. 문제 정의와 기본 수식

### 팩터의 정의

종목 N개, 거래일 T일, 종목마다 시가·종가 같은 변수 M개가 있다고 합시다. 종목 j의 데이터는 M×T 행렬 $x_j \in \mathbb{R}^{M\times T}$입니다. 팩터 f는 이 행렬을 받아 길이 T의 시계열을 내놓는 함수입니다.

$$f : \mathbb{R}^{M\times T} \rightarrow \mathbb{R}^{T}$$

즉 종목 j의 날짜별 팩터 값 $f(x_j)$를 계산하는 규칙입니다. 이 함수의 수식 텍스트 표현을 $s(f)$로 씁니다. 예를 들어 $s(f) =$ "(close − open)/(high − low)"입니다. LLM이 실제로 읽고 쓰는 것은 이 $s(f)$입니다.

### 식 (1): RankIC

$$\text{RankIC}_t(f, r_j) = \text{Corr}(\text{order}^f_{t-1},\ \text{order}^{r_j}_t)$$

$$\gamma(f) = \frac{1}{N}\frac{1}{T}\sum_{j=1}^{N}\sum_{t=1}^{T}\text{RankIC}_t(f, r_j)$$

첫 줄은 t−1일의 팩터 값 순위와 t일의 수익률 순위 사이의 상관입니다. 하루 전 팩터가 다음 날 수익률 순서를 얼마나 맞히는지를 측정합니다. 팩터 값이 높은 종목이 다음 날 실제로 수익률도 높으면 양수, 반대면 음수입니다. 두 번째 줄은 이를 평균한 값으로, 팩터 하나의 예측력을 숫자 하나로 요약한 것이 $\gamma(f)$입니다.

주의할 점이 있습니다. 표준적인 RankIC는 **같은 날짜에 모든 종목의 순위**를 비교하는 횡단면 상관이라 종목 인덱스 j로 합칠 이유가 없습니다. 논문 표기는 종목별로 상관을 구하는 것처럼 써 있어 엄밀하지 않습니다. 실제 구현은 날짜별 횡단면 상관을 평균냈다고 보는 게 자연스럽습니다.

### 식 (2): 팩터 집합의 성능

$$\bar{\gamma}(F) = \mathbb{E}_i[\gamma(f_i)],\quad f_i \in F$$

팩터 여러 개로 이루어진 집합 F의 성능은 각 팩터 RankIC의 단순 평균으로 정의합니다.

### 식 (3): 목표

$$\bar{\gamma}(F_0) \le \bar{\gamma}(F_1) \le \cdots \le \bar{\gamma}(F_m)$$

초기 팩터 집합 $F_0$에서 시작해 마이닝을 한 번 돌릴 때마다 집합의 평균 RankIC가 떨어지지 않게 하는 것이 목표입니다. 반복할수록 집합이 나아져야 한다는 조건입니다.

---

## 3. 방법론

### 3-1. 팩터 간 거리와 상관 (식 4~7)

클러스터링을 하려면 팩터끼리 얼마나 비슷한지를 숫자로 재야 합니다.

**식 (4)**: $\mathbf{p} = \mathbf{w}\mathbf{X}$

모든 종목의 가격 데이터 X를 시가총액 비중 w로 가중평균해서, 시장 전체를 대표하는 가상의 단일 종목 데이터 p를 만듭니다. 사실상 시가총액 가중 지수의 시가·고가·저가·종가 시계열입니다.

**식 (5)**: $\mathbf{v}_i = \dfrac{f_i(\mathbf{p}) - \text{mean}(f_i(\mathbf{p}))}{\text{std}(f_i(\mathbf{p}))}$

팩터 수식 $f_i$를 이 지수 데이터에 적용해 시계열 하나를 얻고, 평균을 빼고 표준편차로 나눠 z-score로 만듭니다. 팩터마다 값의 크기 단위가 다르기 때문에(어떤 건 0.01 단위, 어떤 건 100 단위) 비교 전에 통일하는 단계입니다.

**식 (6)**: $d(f_i, f_j) = \lVert \mathbf{v}_i - \mathbf{v}_j \rVert_2$

두 팩터의 거리는 표준화된 시계열 사이의 유클리드 거리입니다. 날짜마다 두 값의 차이를 제곱해서 모두 더한 뒤 제곱근을 씌운 값입니다.

**식 (7)**: $r(f_i, f_j) = \text{Corr}(\mathbf{v}_i, \mathbf{v}_j)$

두 시계열의 피어슨 상관계수입니다. (논문에 적힌 식은 분모에 제곱근이 빠져 있는 오타가 있습니다. 올바른 분모는 $\sqrt{\sum(v_{it}-\bar v_i)^2}\sqrt{\sum(v_{jt}-\bar v_j)^2}$입니다.)

식 (6)과 (7)의 관계를 짚어두면, 두 벡터가 모두 평균 0·분산 1로 표준화되어 있으면 $d^2 = 2T(1 - r)$이 성립합니다. 거리가 작다는 것과 상관이 높다는 것은 같은 말입니다. 그래서 다음 단계에서 거리 기반 KMeans로 묶으면 결과적으로 상관이 높은 팩터끼리 같은 그룹이 됩니다.

여기서 핵심적인 문제는, 알파 팩터는 원래 종목들 사이의 상대적 순위를 매기는 신호인데 이 방식은 종목이 하나뿐인 지수에 수식을 적용한다는 점입니다. `rank()`처럼 종목 간 비교를 하는 연산자는 종목이 하나면 의미가 사라집니다. 따라서 여기서 잰 상관이 실제로 두 팩터가 같은 종목을 고르는지를 반영한다고 보기 어렵습니다.

### 3-2. CSS: Cross-Sample Selection (식 8~13)

**식 (8)**: $\boldsymbol{\mu}_j = \dfrac{1}{\sum_{c(f_i)=j} 1}\sum_{c(f_i)=j}\mathbf{v}_i$

KMeans의 중심 갱신 식입니다. 순서대로 보면, 먼저 팩터 벡터 중 k개를 랜덤으로 골라 초기 중심으로 둡니다. 각 팩터를 가장 가까운 중심의 그룹에 배정합니다($c(f_i) = \arg\min_j \lVert \mathbf{v}_i - \boldsymbol{\mu}_j\rVert^2$). 그다음 그룹 j에 속한 팩터 벡터들의 평균으로 중심 $\boldsymbol{\mu}_j$를 다시 계산합니다. 분모는 그룹 j의 팩터 개수, 분자는 그 팩터 벡터들의 합입니다. 배정과 갱신을 번갈아 반복합니다.

**식 (9), (10)**: $J = \sum_{i=1}^{k}\sum_{c(f_j)=i}\lVert \mathbf{v}_j - \boldsymbol{\mu}_i\rVert^2,\quad c^* = \arg\min_c J$

J는 각 팩터가 자기 그룹 중심에서 떨어진 거리 제곱의 총합입니다. 그룹이 촘촘할수록 J가 작습니다. 이 J를 최소화하는 배정이 최종 분류 $c^*$입니다. 표준 KMeans 목적함수 그대로입니다.

**식 (11)**: $C_i = \{f_j \mid c^*(f_j) = i\}$

최종적으로 그룹 i에 속한 팩터들의 집합입니다.

**식 (12)**: $FC = [f^1, f^2, \ldots, f^k],\ f^i \in C_i$

각 그룹에서 팩터를 하나씩 랜덤으로 뽑아 k개짜리 조합을 만듭니다. 그룹마다 하나씩이니 조합 안의 팩터들은 서로 상관이 낮습니다.

**식 (13)**: $S = [s(f^{i_1}), \ldots, s(f^{i_l})],\ l \le k$

그 조합에서 다시 l개를 골라 수식 텍스트로 바꾼 것이 프롬프트에 들어갈 예시 S입니다. 실험에서는 l=2입니다.

### 3-3. CoE: Chain-of-Experience (식 14~15)

**식 (14)**: 초기 체인

$$\text{CoE}^0_i = s(f^{(i)}_1) \rightarrow s(f^{(i)}_2) \rightarrow \cdots \rightarrow s(f^{(i)}_{p_i}),\quad \gamma(f^{(i)}_1) \le \cdots \le \gamma(f^{(i)}_{p_i})$$

그룹 $C_i$ 안의 팩터 $p_i$개를 RankIC가 낮은 것부터 높은 것 순으로 정렬해 화살표로 연결합니다. 실제 순서대로 발전한 게 아니라 성능 순으로 줄 세운 것이지만, LLM에게는 "이쪽 방향으로 갈수록 좋아진다"는 경로처럼 보이게 됩니다. 프롬프트의 `improve_path`가 이것입니다.

**식 (15)**: 체인 갱신

$$\text{CoE}^k_i = s(f^{(i)}_1) \rightarrow \cdots \rightarrow s(f^{(i)}_h) \rightarrow s(f^{*(i)}),\quad r(f^{(i)}_h, f^{*(i)}) \ge r(f^{(i)}_j, f^{*(i)})\ \forall j$$

이전 체인을 보고 LLM이 만든 새 팩터 $f^*$가 체인 안의 모든 팩터보다 RankIC가 높을 때만 체인을 갱신합니다. 과정은 이렇습니다. 체인의 각 팩터와 $f^*$의 상관(식 7)을 계산해 가장 상관이 높은 팩터 $f_h$를 찾습니다. 아래 조건식이 바로 "$f_h$는 $f^*$와의 상관이 체인 내 어떤 팩터보다도 크거나 같다"는 뜻입니다. 그다음 체인을 처음부터 $f_h$까지만 남기고 그 뒤에 $f^*$를 붙입니다.

$f_h$가 체인의 맨 끝이면 아무것도 잘리지 않으니 기존 체인을 한 칸 연장한 셈입니다. $f_h$가 중간이면 그 뒤의 팩터들이 잘려나가고 $f^*$로 갈라진 새 경로가 됩니다. 직관적으로는 "새 팩터가 체인의 어느 지점에서 파생된 것과 가장 닮았는지"를 찾아 그 지점에서 가지를 친다는 것입니다.

### 3-4. 전체 알고리즘 (Algorithm 1)

먼저 식 (14)로 그룹별 초기 체인을 만듭니다. 그다음 최대 m번 반복하는데, 각 반복에서 이전 팩터 집합을 클러스터링하고(식 11), CSS로 예시를 고르고(식 13), 각 예시마다 관련 체인을 찾아 예시와 체인을 합친 프롬프트를 LLM에 넣어 새 팩터를 받습니다. 새 팩터가 체인 내 최고 RankIC보다 좋으면 식 (15)로 체인을 갱신하고, 새 팩터를 팩터 집합에 추가합니다. 집합의 평균 RankIC가 이전보다 오르고 팩터 수가 15개 이상이면 그 반복을 끝냅니다.

부록 D의 실제 프롬프트는 "잘 작동하는 알파들을 줄 테니 이를 융합해 입력과 다른 새 알파를 만들고, improve_path의 방향을 참고하라"는 지시와 Alpha101 연산자 정의로 구성됩니다.

---

## 4. 실험 설정

초기 팩터는 Alpha101 중 38개(본문 기준, 부록 B에는 71개가 나열돼 있어 불일치)입니다. 클러스터 수는 7로, 사람이 금융 지식으로 분류해본 뒤 정했습니다. 예시 수 l=2, 최소 팩터 수 u=15이고, LLM은 text-davinci-002를 temperature 0으로 사용했습니다.

데이터는 S&P500 구성 종목의 일별 시가·고가·저가·종가·거래량, 기간은 2015~2021년입니다. 베이스라인 모델들은 2015~2019년을 학습에, 2020년을 검증에, 2021년을 테스트에 썼습니다. FAMA는 2020년 6월~12월만 학습·검증에 썼고, 이를 "학습 데이터의 10%만 사용"이라고 표현합니다.

비교 대상은 기호 계열로 Alpha101 원본, GP, CSS·CoE 없이 LLM만 쓴 버전, 신경망 계열로 DTransformer, ALSTM, FactorVAE입니다.

---

## 5. 결과와 관련 수식

### 식 (16): RankICIR

$$\text{RankICIR} = \mathbb{E}_f\left[\mathbb{E}_j\left[\frac{\gamma(f)}{\sigma_{\text{RankIC}_t(f,r_j)}}\right]\right]$$

RankIC의 평균을 RankIC의 표준편차로 나눈 값입니다. 평균 RankIC가 같아도 날마다 들쭉날쭉하면 실전에서 쓰기 어렵기 때문에, 예측력의 안정성을 보는 지표입니다. 샤프 비율의 RankIC 버전이라고 보면 됩니다.

### 예측 성능 (Table 1)

FAMA(CI-7), 즉 CSS와 CoE를 함께 쓰고 7번 반복한 버전이 RankIC 0.054, RankICIR 0.485로 가장 높습니다. 최강 베이스라인 FactorVAE는 0.048, 0.379입니다. 차이는 RankIC 0.006, RankICIR 0.106입니다.

소거 실험(ablation)을 보면 LLM만 쓰면 0.015, CSS만 쓰면 0.023, CoE만 1회 쓰면 0.016으로 각각은 효과가 작고, 둘을 함께 쓰며 반복을 늘릴수록 올라갑니다(3회 0.030, 7회 0.054).

### 파라미터 실험 (Figure 3, 4)

CoE 반복 횟수를 늘리면 성능이 대체로 올라갑니다. CSS 예시 수는 3개까지는 늘릴수록 좋아지다가 그 이상이면 떨어지고, 7개에서는 RankIC가 음수가 됩니다. 체인에서 노드를 하나씩 지워보는 실험에서는 맨 앞(성능이 가장 낮은) 노드를 지우면 오히려 성능이 오르고, 맨 끝(최고 성능) 노드를 지우면 크게 떨어집니다. 저자들은 저성능 노드가 너무 많으면 방해가 되니 체인 길이를 조절해야 한다고 해석합니다.

### 식 (17): 포트폴리오 팩터 가중치

$$w_i = \frac{\text{RankIC}^{past}_i}{\sum_{i=0}^{n}\text{RankIC}^{past}_i}$$

팩터 여러 개를 하나의 점수로 합칠 때, 2020년 6~12월의 평균 RankIC에 비례해 가중치를 줍니다. 과거 예측력이 좋았던 팩터에 더 많은 비중을 두는 단순한 방식입니다. RankIC가 음수인 팩터가 있으면 가중치가 음수가 되는데, 앞서 RankIC 0.01 이상인 팩터만 골랐으니 그 문제는 피해갑니다. 합성 점수 상위 20% 종목을 매수하고 다음 날 매도합니다.

### 식 (18)~(20): 성과 지표

**식 (18)**: $AR = (1+R)^{252/N} - 1$

누적수익률 R을 연환산한 값입니다. N일 동안 R만큼 벌었으니 1년(252거래일) 기준으로는 $(1+R)$을 $252/N$ 제곱한 만큼 불어난다는 복리 환산입니다.

**식 (19)**: $Vol = \sigma_p \sqrt{252}$

일간 수익률 표준편차를 연환산합니다. 일간 수익률이 서로 독립이면 분산은 기간에 비례해 커지고, 표준편차는 기간의 제곱근에 비례하기 때문에 $\sqrt{252}$를 곱합니다.

**식 (20)**: $SR = \dfrac{R_p - R_f}{\sigma_p}\sqrt{252}$

일간 초과수익의 평균을 일간 표준편차로 나눈 뒤 연환산합니다. 무위험 이자율 $R_f$는 0으로 두었습니다. 표에서 667.2%로 적힌 것은 샤프 6.67을 퍼센트로 표기한 것입니다.

### 투자 시뮬레이션 (Table 2)

FAMA가 연수익률 38.4%, 변동성 4.9%, 샤프 6.67로 모든 비교 대상을 앞섭니다. S&P500 전체 보유는 26.3%, 11.5%, 2.09입니다. 저자들은 기호 방식의 낮은 변동성과 신경망 방식의 높은 수익률을 동시에 얻었다고 주장합니다.

---

## 6. 읽을 때 유의할 점

앞서 말씀드린 내용을 요약하면 다음과 같습니다. 해석 가능성은 출력이 수식이라는 수준이고 경제적 의미 부여는 없습니다. 팩터 간 상관을 종목 하나짜리 지수 시계열에서 계산해 횡단면 팩터의 유사성을 제대로 반영하지 못합니다. FactorVAE 대비 개선폭 0.006은 표준편차(0.008~0.010) 안이라 유의하다고 보기 어렵습니다. 롱온리 전략이 지수 변동성의 절반 이하라는 결과는 구조적으로 납득이 어렵고, 거래비용은 반영되지 않았으며, 테스트는 2021년 한 해뿐입니다. 7개월 데이터로 팩터를 고른 점도 과적합 위험을 키웁니다.

방법론에서 가져갈 만한 것은 "상관 낮은 예시로 다양성 확보"와 "성능 순 경로를 프롬프트에 넣어 개선 방향 제시"라는 두 가지 프롬프트 설계 아이디어이고, 성과 수치는 재현 없이는 참고하기 어렵습니다.